# Stage 3 Time Naive Baselines BPI2012 Colab 07

This notebook computes simple next-time naive baselines for Stage 3 using the
same leave-one-out split logic as the SASRec experiments.

Naive baselines:
- global mean
- global median
- current activity mean
- prefix length mean

Evaluation metrics:
- MAE
- RMSE
- Median AE


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only_stage3_v2'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10_v2'
MULTITASK_BASELINE_W01_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_w01_ndcg10_v2'
MULTITASK_ATTNBIAS_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_ndcg10_v2'
MULTITASK_ATTNBIAS_W01_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_attention_bias_multitask_w01_ndcg10_v2'

print('DATA_DIR:', DATA_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2


In [3]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
Cloning into 'time-aware-behavior-prediction'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 409 (delta 269), reused 270 (delta 133), pack-reused 0 (from 0)
Receiving objects: 100% (409/409), 12.29 MiB | 7.43 MiB/s, done.
Resolving deltas: 100% (269/269), done.
/content/time-aware-behavior-prediction
Already up to date.


In [4]:
%cd /content/time-aware-behavior-prediction
!python scripts/regenerate_stage3_processed_dataset.py --output-dir "$DATA_DIR" --backup-existing
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only_stage3_v2
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only_stage3_v2


/content/time-aware-behavior-prediction
[info] moved existing output to backup: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2__backup_20260602_053219
[ok] regenerated Stage 3 processed dataset at: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2
[ok] metadata written to: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only_stage3_v2/stage3_dataset_metadata.json
{
  "timestamp": 0,
  "delta_prev_seconds": 0,
  "delta_start_seconds": 0,
  "delta_next_seconds": 0
}
events_complete_only_filtered.csv  sasrec_interactions.csv	 user_map.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   stage3_dataset_metadata.json


In [5]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


## Build train / valid / test rows with the same split logic

The SASRec split is leave-one-out:
- train = sequence except last 2 events
- valid target = second last event
- test target = last event

For next-time:
- train samples use every current event in train with its `delta_next_seconds`
- valid sample uses the last train event -> valid event gap
- test sample uses the valid event -> test event gap


In [6]:
interactions_path = 'data/processed/bpi2012_complete_only_stage3_v2/sasrec_interactions.txt'
time_features_path = 'data/processed/bpi2012_complete_only_stage3_v2/events_encoded_time_features.csv'

interactions = pd.read_csv(
    interactions_path,
    sep=' ',
    header=None,
    names=['user_id', 'item_id'],
)

timef = pd.read_csv(time_features_path)
timef['user_id'] = pd.to_numeric(timef['user_id']).astype(int)
timef['event_idx'] = pd.to_numeric(timef['event_idx']).astype(int)
timef['item_id'] = pd.to_numeric(timef['item_id']).astype(int)
timef['delta_next_seconds'] = pd.to_numeric(timef['delta_next_seconds'], errors='coerce')

interactions['user_id'] = interactions['user_id'].astype(int)
interactions['item_id'] = interactions['item_id'].astype(int)

# Sanity check alignment with processed file order
merged = interactions.copy()
merged['event_idx'] = merged.groupby('user_id').cumcount()
check = merged.merge(
    timef[['user_id', 'event_idx', 'item_id', 'activity', 'delta_next_seconds']],
    on=['user_id', 'event_idx', 'item_id'],
    how='left',
)
if check['activity'].isna().any():
    raise ValueError('Failed to align interactions with time feature rows.')

def build_split_rows(df):
    train_rows = []
    valid_rows = []
    test_rows = []

    for user_id, g in df.groupby('user_id', sort=True):
        g = g.sort_values('event_idx').reset_index(drop=True)
        n = len(g)
        if n < 4:
            continue

        train_g = g.iloc[:-2].reset_index(drop=True)
        valid_g = g.iloc[[-2]].reset_index(drop=True)
        test_g = g.iloc[[-1]].reset_index(drop=True)

        # train rows: each current event in train predicts its next gap
        for i, row in train_g.iterrows():
            train_rows.append({
                'split': 'train',
                'user_id': user_id,
                'event_idx': int(row['event_idx']),
                'current_activity': row['activity'],
                'prefix_length': int(i + 1),
                'y_true': float(row['delta_next_seconds']),
            })

        # valid row: current event is last train event
        last_train = train_g.iloc[-1]
        valid_rows.append({
            'split': 'valid',
            'user_id': user_id,
            'event_idx': int(last_train['event_idx']),
            'current_activity': last_train['activity'],
            'prefix_length': int(len(train_g)),
            'y_true': float(last_train['delta_next_seconds']),
        })

        # test row: current event is valid event (second last original)
        valid_event = valid_g.iloc[0]
        test_rows.append({
            'split': 'test',
            'user_id': user_id,
            'event_idx': int(valid_event['event_idx']),
            'current_activity': valid_event['activity'],
            'prefix_length': int(len(train_g) + 1),
            'y_true': float(valid_event['delta_next_seconds']),
        })

    return (
        pd.DataFrame(train_rows),
        pd.DataFrame(valid_rows),
        pd.DataFrame(test_rows),
    )

train_df, valid_df, test_df = build_split_rows(check)

print('train rows:', len(train_df))
print('valid rows:', len(valid_df))
print('test rows:', len(test_df))

train_df.head()


train rows: 134903
valid rows: 9658
test rows: 9658


,split,user_id,event_idx,current_activity,prefix_length,y_true
0,train,1,0,A_SUBMITTED,1,0.334
1,train,1,1,A_PARTLYSUBMITTED,2,53.026
2,train,1,2,A_PREACCEPTED,3,39785.402
3,train,1,3,A_ACCEPTED,4,145.935
4,train,1,4,A_FINALIZED,5,-0.000


## Compute naive baselines

Baselines:
- `global_mean`
- `global_median`
- `activity_mean`
- `prefix_len_mean`


In [7]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def median_ae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.median(np.abs(y_true - y_pred)))

global_mean = float(train_df['y_true'].mean())
global_median = float(train_df['y_true'].median())
activity_mean_map = train_df.groupby('current_activity')['y_true'].mean().to_dict()
prefix_mean_map = train_df.groupby('prefix_length')['y_true'].mean().to_dict()

def predict_global_mean(df):
    return np.full(len(df), global_mean, dtype=float)

def predict_global_median(df):
    return np.full(len(df), global_median, dtype=float)

def predict_activity_mean(df):
    fallback = global_mean
    return np.array([activity_mean_map.get(act, fallback) for act in df['current_activity']], dtype=float)

def predict_prefix_mean(df):
    fallback = global_mean
    return np.array([prefix_mean_map.get(int(pl), fallback) for pl in df['prefix_length']], dtype=float)

baseline_predictors = {
    'global_mean': predict_global_mean,
    'global_median': predict_global_median,
    'activity_mean': predict_activity_mean,
    'prefix_len_mean': predict_prefix_mean,
}

rows = []
for split_name, split_df in [('valid', valid_df), ('test', test_df)]:
    for baseline_name, predictor in baseline_predictors.items():
        pred = predictor(split_df)
        rows.append({
            'split': split_name,
            'baseline': baseline_name,
            'mae': mae(split_df['y_true'], pred),
            'rmse': rmse(split_df['y_true'], pred),
            'median_ae': median_ae(split_df['y_true'], pred),
        })

baseline_results = pd.DataFrame(rows)
baseline_results


,split,baseline,mae,rmse,median_ae
0,valid,global_mean,114284.763840,288388.780302,71235.512711
1,valid,global_median,67245.870600,296052.587905,198.116000
2,valid,activity_mean,83428.490744,277060.289915,6854.339538
3,valid,prefix_len_mean,113639.332890,285881.712454,67014.783253
4,test,global_mean,80778.113573,113778.214930,71232.032711
5,test,global_median,13491.763875,98806.978174,194.680500
6,test,activity_mean,21945.296709,82838.049387,73.489166
7,test,prefix_len_mean,76546.064085,116432.880525,67007.703253


## Convert seconds to readable units


In [8]:
readable = baseline_results.copy()
for col in ['mae', 'rmse', 'median_ae']:
    readable[f'{col}_hours'] = readable[col] / 3600.0
    readable[f'{col}_minutes'] = readable[col] / 60.0

readable


,split,baseline,mae,rmse,median_ae,mae_hours,mae_minutes,rmse_hours,rmse_minutes,median_ae_hours,median_ae_minutes
0,valid,global_mean,114284.763840,288388.780302,71235.512711,31.745768,1904.746064,80.107995,4806.479672,19.787642,1187.258545
1,valid,global_median,67245.870600,296052.587905,198.116000,18.679409,1120.764510,82.236830,4934.209798,0.055032,3.301933
2,valid,activity_mean,83428.490744,277060.289915,6854.339538,23.174581,1390.474846,76.961192,4617.671499,1.903983,114.238992
3,valid,prefix_len_mean,113639.332890,285881.712454,67014.783253,31.566481,1893.988882,79.411587,4764.695208,18.615218,1116.913054
4,test,global_mean,80778.113573,113778.214930,71232.032711,22.438365,1346.301893,31.605060,1896.303582,19.786676,1187.200545
5,test,global_median,13491.763875,98806.978174,194.680500,3.747712,224.862731,27.446383,1646.782970,0.054078,3.244675
6,test,activity_mean,21945.296709,82838.049387,73.489166,6.095916,365.754945,23.010569,1380.634156,0.020414,1.224819
7,test,prefix_len_mean,76546.064085,116432.880525,67007.703253,21.262796,1275.767735,32.342467,1940.548009,18.613251,1116.795054


## Compare with Stage 3 model results

This block extracts saved Stage 3 multitask results and compares their
test next-time metrics with the naive baselines.


In [9]:
def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'seed': config.get('seed'),
            'time_loss_weight': config.get('time_loss_weight'),
        }
        for group_name in ['best_test_at_best_valid']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)

mt_w10_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)
mt_anchor_w01_df = rebuild_df(MULTITASK_BASELINE_W01_OUTPUT_DIR)
mt_attnbias_w10_df = rebuild_df(MULTITASK_ATTNBIAS_OUTPUT_DIR)
mt_attnbias_w01_df = rebuild_df(MULTITASK_ATTNBIAS_W01_OUTPUT_DIR)

model_rows = []

for _, r in mt_w10_df[mt_w10_df['run_name'].isin([
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
])].iterrows():
    model_rows.append({
        'model': 'anchor_multi_task_w1.0',
        'mae': r.get('best_test_at_best_valid_task_time_mae'),
        'rmse': r.get('best_test_at_best_valid_task_time_rmse'),
        'median_ae': r.get('best_test_at_best_valid_task_time_median_ae'),
    })

for _, r in mt_anchor_w01_df[mt_anchor_w01_df['run_name'].isin([
    'multitask_anchor_ml20_w01_s42',
    'multitask_anchor_ml20_w01_s2024',
    'multitask_anchor_ml20_w01_s7',
])].iterrows():
    model_rows.append({
        'model': 'anchor_multi_task_w0.1',
        'mae': r.get('best_test_at_best_valid_task_time_mae'),
        'rmse': r.get('best_test_at_best_valid_task_time_rmse'),
        'median_ae': r.get('best_test_at_best_valid_task_time_median_ae'),
    })

for _, r in mt_attnbias_w10_df[mt_attnbias_w10_df['run_name'].isin([
    'multitask_attnbias_dstart_ml20_b9_s42',
    'multitask_attnbias_dstart_ml20_b9_s2024',
    'multitask_attnbias_dstart_ml20_b9_s7',
])].iterrows():
    model_rows.append({
        'model': 'anchor_attnbias_multi_task_w1.0',
        'mae': r.get('best_test_at_best_valid_task_time_mae'),
        'rmse': r.get('best_test_at_best_valid_task_time_rmse'),
        'median_ae': r.get('best_test_at_best_valid_task_time_median_ae'),
    })

for _, r in mt_attnbias_w01_df[mt_attnbias_w01_df['run_name'].isin([
    'multitask_attnbias_dstart_ml20_b9_w01_s42',
    'multitask_attnbias_dstart_ml20_b9_w01_s2024',
    'multitask_attnbias_dstart_ml20_b9_w01_s7',
])].iterrows():
    model_rows.append({
        'model': 'anchor_attnbias_multi_task_w0.1',
        'mae': r.get('best_test_at_best_valid_task_time_mae'),
        'rmse': r.get('best_test_at_best_valid_task_time_rmse'),
        'median_ae': r.get('best_test_at_best_valid_task_time_median_ae'),
    })

model_df = pd.DataFrame(model_rows)
model_summary = model_df.groupby('model')[['mae', 'rmse', 'median_ae']].agg(['mean', 'std'])
model_summary


mae                       rmse  \
                                         mean          std          mean   
model                                                                      
anchor_attnbias_multi_task_w0.1  13225.002521   258.643906  83186.237224   
anchor_attnbias_multi_task_w1.0  11937.164159   375.675496  71007.381158   
anchor_multi_task_w0.1           14942.777351  1499.612136  75814.317546   
anchor_multi_task_w1.0           12674.034845  1361.526740  72600.274475   

                                               median_ae              
                                         std        mean         std  
model                                                                 
anchor_attnbias_multi_task_w0.1  4904.586252   41.718635   35.310037  
anchor_attnbias_multi_task_w1.0   753.747538  100.579255   73.859784  
anchor_multi_task_w0.1            120.720828  194.508229  198.534642  
anchor_multi_task_w1.0           5986.168541   71.948215   56.728439

Interpretation guide:

- if a multitask model beats `global_mean` / `global_median`, it is at least better than trivial prediction
- if it also beats `activity_mean` and `prefix_len_mean`, the next-time head is learning something beyond simple heuristics
- `median_ae` reflects typical-case error
- `mae` and `rmse` reflect tail sensitivity
